In [23]:
import requests
import json
import pandas as pd
from datetime import datetime

In [18]:
def mongol_bank_khansh(date):
    url = f"https://www.mongolbank.mn/mn/currency-rates/data?startDate={date}&endDate={date}"
    response = requests.post(url)
    data = response.json()
    return pd.DataFrame(data['data'])

In [19]:
mongol_bank_khansh('2026-01-01')

,RATE_DATE,USD,EUR,JPY,CHF,SEK,GBP,BGN,HUF,EGP,...,PLN,UAH,NOK,NPR,ZAR,TRY,VND,XAU,XAG,SDR
0,2026-01-01,"3,556.81","4,169.83",22.71,"4,478.76",385.82,"4,778.40","2,138.98",10.83,74.65,...,987.45,83.98,353.11,24.74,214.29,82.79,0.14,"15,348,115.32","255,887.94","4,875.36"


In [29]:
def mongol_bank_khansh(date: str) -> pd.DataFrame:
    # 1. Validate date format (YYYY-MM-DD)
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        raise ValueError("Invalid date format. Expected YYYY-MM-DD")

    url = f"https://www.mongolbank.mn/mn/currency-rates/data?startDate={date}&endDate={date}"

    try:
        response = requests.post(url)
    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Request failed: {e}")

    # 2. Check status code
    if 400 <= response.status_code < 500:
        raise Exception(f"Client error ({response.status_code}): Check request parameters")
    elif 500 <= response.status_code < 600:
        raise Exception(f"Server error ({response.status_code}): MongolBank API issue")

    if response.status_code != 200:
        raise Exception(f"Unexpected status code: {response.status_code}")

    # 3. Validate JSON response
    try:
        data = response.json()
    except ValueError:
        return "API дуудхад алдаа гарлаа, хөгжүүлэгчтэй холбогдоорой"

    if not data.get("success", False):
        return "API дуудхад алдаа гарлаа. Түр хүлээгээд дахиад асуугаарай"

    if "data" not in data:
        return "Тухайн өдрийн ханшийн мэдээлэл байхгүй байна"

    return pd.DataFrame(data["data"])

In [30]:
mongol_bank_khansh('2026-01-01')

,RATE_DATE,USD,EUR,JPY,CHF,SEK,GBP,BGN,HUF,EGP,...,PLN,UAH,NOK,NPR,ZAR,TRY,VND,XAU,XAG,SDR
0,2026-01-01,"3,556.81","4,169.83",22.71,"4,478.76",385.82,"4,778.40","2,138.98",10.83,74.65,...,987.45,83.98,353.11,24.74,214.29,82.79,0.14,"15,348,115.32","255,887.94","4,875.36"


In [31]:
mongol_bank_khansh('2026-10-01')

'API дуудхад алдаа гарлаа. Түр хүлээгээд дахиад асуугаарай'

In [32]:
mongol_bank_khansh('1900-10-01')

'API дуудхад алдаа гарлаа. Түр хүлээгээд дахиад асуугаарай'

In [33]:
import ast
import operator

OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg
}

def eval_expr(expr: str):
    def _eval(node):
        if isinstance(node, ast.Num):
            return node.n
        
        elif isinstance(node, ast.BinOp):
            if type(node.op) not in OPS:
                return "Unsupported operator"
            return OPS[type(node.op)](_eval(node.left), _eval(node.right))
        
        elif isinstance(node, ast.UnaryOp):
            if type(node.op) not in OPS:
                return "Unsupported unary operator"
            return OPS[type(node.op)](_eval(node.operand))
        
        else:
            return "Unsupported expression"

    try:
        parsed = ast.parse(expr, mode='eval')
        result = _eval(parsed.body)

        # Хэрвээ дотор нь алдаа string буцсан бол дамжуулна
        if isinstance(result, str):
            return result

        return result

    except ZeroDivisionError:
        return "Division by zero"
    except Exception:
        return "Invalid expression"

In [34]:
exp = "(2+4)*5"

In [36]:
exp1 = "2000-01-01"

In [38]:
eval_expr(exp1)

'Invalid expression'

In [45]:
import re

def handle_user_query(user_query: str):
    if not isinstance(user_query, str):
        return "Invalid input"

    query = user_query.strip()
    query_lower = query.lower()

    # 1. Contact шалгах
    if any(word in query_lower for word in ["утас", "contact", "холбоо барих", "holbodgoh medeelel"]):
        return "Манай холбоо барих утасны дугаар: 99887766"

    # 2. Location шалгах
    if any(word in query_lower for word in ["байршил", "location"]):
        return "Манай байршил: Galaxy tower 7 давхар, 705 тоот"

    # 3. Date format шалгах (YYYY-MM-DD)
    date_pattern = r"^\d{4}-\d{2}-\d{2}$"
    if re.match(date_pattern, query):
        try:
            return mongol_bank_khansh(query)
        except Exception as e:
            return str(e)

    # 4. Math expression гэж үзэх оролдлого
    # (зөвхөн math тэмдэгтүүд байгаа эсэхийг шалгана)
    math_pattern = r"^[\d+\-*/().\s^]+$"
    if re.match(math_pattern, query):
        try:
            return eval_expr(query)
        except Exception as e:
            return str(e)

    # 5. Default → 그대로 буцаана
    return user_query

In [46]:
user_query = "2026-01-01"

In [47]:
handle_user_query(user_query)

,RATE_DATE,USD,EUR,JPY,CHF,SEK,GBP,BGN,HUF,EGP,...,PLN,UAH,NOK,NPR,ZAR,TRY,VND,XAU,XAG,SDR
0,2026-01-01,"3,556.81","4,169.83",22.71,"4,478.76",385.82,"4,778.40","2,138.98",10.83,74.65,...,987.45,83.98,353.11,24.74,214.29,82.79,0.14,"15,348,115.32","255,887.94","4,875.36"


In [48]:
user_query = "2026-1-1"
handle_user_query(user_query)

/var/folders/wb/__pjzmzj3f304_6w13kj39880000gn/T/ipykernel_15868/4182651054.py:15: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):
/var/folders/wb/__pjzmzj3f304_6w13kj39880000gn/T/ipykernel_15868/4182651054.py:16: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  return node.n


2024

In [49]:
user_query = "location haana bile"
handle_user_query(user_query)

'Манай байршил: Galaxy tower 7 давхар, 705 тоот'

In [50]:
user_query = "holbodgoh medeelel"
handle_user_query(user_query)

'Манай холбоо барих утасны дугаар: 99887766'

In [51]:
user_query = "holbodgoh medeelel, bairshiin medeelel ugchee"
handle_user_query(user_query)

'Манай холбоо барих утасны дугаар: 99887766'

In [ ]:
'''Манай холбоо барих утасны дугаар: 99887766
Манай байршил: Galaxy tower 7 давхар, 705 тоот'''

In [2]:
import xlrd
print(xlrd.__version__)

2.0.1


In [3]:
import pandas as pd
print(pd.__version__)

2.2.3


In [4]:
import streamlit as st
print(st.__version__)

1.41.1


In [5]:
import os
os.getcwd()

'/Users/suvdaa/Documents/Others/Way/Streamlit'

In [9]:
user_query = "Хол"

In [12]:
keywords = ["утас", "contact", "холбоо барих"]

if any(k in user_query.lower() for k in keywords):
    print("Та холбоо барих хэсэг асууж байна")
else:
    print("Өөр юм асуу")

Өөр юм асуу


In [13]:
import re
import ast
import operator
import requests

# --- 1. Keyword intent ---
CONTACT_KEYWORDS = ["утас", "contact", "холбоо барих"]
LOCATION_KEYWORDS = ["байршил", "location"]

# --- 2. Safe math evaluator ---
ops = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}

def safe_eval(node):
    if isinstance(node, ast.Num):
        return node.n
    elif isinstance(node, ast.BinOp):
        return ops[type(node.op)](safe_eval(node.left), safe_eval(node.right))
    else:
        raise ValueError("Invalid expression")

def is_math_expression(s):
    return re.fullmatch(r"[0-9+\-*/().\s]+", s) is not None

def calculate(expr):
    tree = ast.parse(expr, mode='eval')
    return safe_eval(tree.body)

# --- 3. Date format шалгах ---
def is_date(s):
    return re.fullmatch(r"\d{4}-\d{2}-\d{2}", s) is not None

# --- 4. Монгол банк API (жишээ endpoint) ---
def get_exchange_rate(date):
    url = f"https://api.mongolbank.mn/api/exchange-rate?start={date}&end={date}"
    try:
        res = requests.get(url)
        data = res.json()
        return data
    except Exception:
        return "Ханшийн мэдээлэл авахад алдаа гарлаа"

# --- MAIN ROUTER ---
def handle_query(user_query: str):
    q = user_query.lower()

    # 1. Contact
    if any(k in q for k in CONTACT_KEYWORDS):
        return "Манай холбоо барих утасны дугаар: 99887766"

    # 2. Location
    if any(k in q for k in LOCATION_KEYWORDS):
        return "Манай байршил: Galaxy tower 7 давхар, 705 тоот"

    # 3. Math
    if is_math_expression(q):
        try:
            return f"Хариу: {calculate(q)}"
        except:
            return "Бодож чадсангүй"

    # 4. Date → API
    if is_date(q):
        return get_exchange_rate(q)

    # 5. Default
    return user_query


In [14]:
# --- TEST ---
queries = [
    "утас хэд вэ?",
    "location хаана вэ?",
    "(2+3)*2",
    "2024-01-15",
    "сайн байна уу"
]

for q in queries:
    print(q, "→", handle_query(q))

утас хэд вэ? → Манай холбоо барих утасны дугаар: 99887766
location хаана вэ? → Манай байршил: Galaxy tower 7 давхар, 705 тоот
(2+3)*2 → Хариу: 10
2024-01-15 → Бодож чадсангүй
сайн байна уу → сайн байна уу


/var/folders/wb/__pjzmzj3f304_6w13kj39880000gn/T/ipykernel_15868/3539266012.py:19: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):
/var/folders/wb/__pjzmzj3f304_6w13kj39880000gn/T/ipykernel_15868/3539266012.py:20: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  return node.n
